# Phase 1: Scalarized Bayesian Optimization for 200 MeV Electron Injector Linac

This interactive notebook demonstrates **Phase 1 Scalarized Bayesian Optimization (BO)** for the 200 MeV electron injector linac simulation. Scalarized BO combines multiple beam quality objectives into a single scalar merit function using weight combinations:

$$f(\mathbf{x}) = w_1 \cdot \varepsilon_{n,x} + w_2 \cdot \varepsilon_{n,y} + w_3 \cdot \sigma_E$$

A single Gaussian Process surrogate (`SingleTaskGP`) with Matérn 5/2 ARD kernel is fitted to the scalarized objective, and candidates are selected using `qLogNoisyExpectedImprovement` (`qLogNEI`).

In [ ]:
# Useful for interactive debugging and live code updates
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is in path
import sys
sys.path.insert(0, str(Path("..").resolve()))

from mobo_linac.config import load_config
from mobo_linac.execution.parallel import BatchEvaluator
from mobo_linac.cli import run_scalarized

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Environment & Binary Setup

Verify environment variable settings for local ASTRA binaries:

In [ ]:
project_root = Path("..").resolve()
bin_dir = project_root / "bin"
astra_bin = bin_dir / "astra"
generator_bin = bin_dir / "generator"

os.environ["ASTRA_BIN"] = str(astra_bin)
os.environ["GENERATOR_BIN"] = str(generator_bin)

print(f"ASTRA_BIN: {os.environ.get('ASTRA_BIN')}")
print(f"GENERATOR_BIN: {os.environ.get('GENERATOR_BIN')}")
print(f"Binary exists: {astra_bin.exists()}")

## 2. Load Configuration

Load central YAML configuration parameters (6D decision variables, beam constraints, reference points):

In [ ]:
config_path = project_root / "configs" / "mobo_200mev.yaml"
config = load_config(config_path)
print(f"Loaded config: {config.name}")
print(f"Design Variables: {len(config.design_variables)}")
for p in config.design_variables:
    print(f"  - {p.name} [{p.astra_name}]: bounds = [{p.bounds[0]}, {p.bounds[1]}] {p.unit}")

## 3. Configure and Execute Scalarized BO Campaign

Execute Phase 1 scalarized BO with custom weight vectors (e.g. equal weights `[1.0, 1.0, 1.0]`):

In [ ]:
class ScalarizedBOArgs:
    config = str(config_path)
    n_iterations = 10
    batch_size = 4
    num_initial_samples = 8
    num_workers = 4
    weights = [1.0, 1.0, 1.0]
    seed = 42
    output_dir = str(project_root / "results" / "phase1_scalarized_demo")

args = ScalarizedBOArgs()
print(f"Running Phase 1 Scalarized BO demo to {args.output_dir}...")
# Uncomment to run simulation:
# run_scalarized(args)

## 4. Analyze Results & Trade-off Plots

Load results CSVs and plot objective progression:

In [ ]:
results_dir = project_root / "results" / "phase1_scalarized_demo"
if (results_dir / "train_Y.csv").exists():
    df_y = pd.read_csv(results_dir / "train_Y.csv")
    print("Loaded evaluations:", len(df_y))
    print(df_y.head())
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(df_y['emittance_x'], 'b-o', label='\epsilon_{n,x}')
    ax[0].set_title('Horizontal Emittance [mm mrad]')
    ax[0].grid(True)
    
    ax[1].plot(df_y['emittance_y'], 'g-o', label='\epsilon_{n,y}')
    ax[1].set_title('Vertical Emittance [mm mrad]')
    ax[1].grid(True)
    
    ax[2].plot(df_y['energy_spread'], 'r-o', label='\sigma_E')
    ax[2].set_title('RMS Energy Spread [MeV]')
    ax[2].grid(True)
    plt.tight_layout()
    plt.show()
else:
    print(f"No run output found at {results_dir}. Execute the run cell above first.")